# 7. Modelado Predictivo

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

RANDOM_STATE = 42
DATA_DIR = "../data"
MODEL_CSV = f"{DATA_DIR}/processed/gb_videos_model.csv"
FIG_DIR = "../reports/figures"
os.makedirs(FIG_DIR, exist_ok=True)
sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv(MODEL_CSV)
print("Shape:", df.shape)

## 7.1 Definir X e y

In [ ]:
# Variable objetivo y predictoras. Usamos log1p(views) para estabilizar la varianza.
X = df.drop(columns="views")
y = np.log1p(df["views"])
print("Features:", X.shape[1])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

## 7.2 Entrenar y comparar modelos

In [ ]:
# Definición de modelos a comparar (incluye baseline)
modelos = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, max_depth=15, n_jobs=-1, random_state=RANDOM_STATE
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200, max_depth=5, random_state=RANDOM_STATE
    ),
}
if HAS_XGB:
    modelos["XGBoost"] = XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        n_jobs=-1, random_state=RANDOM_STATE, verbosity=0
    )

def evaluar(y_true, y_pred, nombre):
    return {
        "model": nombre,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

resultados = []
# Baseline: media del target
pred_base = np.full_like(y_test, y_train.mean(), dtype=float)
resultados.append(evaluar(y_test, pred_base, "Baseline (media)"))

modelos_entrenados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    resultados.append(evaluar(y_test, pred, nombre))
    modelos_entrenados[nombre] = modelo

metrics = pd.DataFrame(resultados).round(4)
display(metrics.sort_values("MAE"))

## 7.3 Validación cruzada del mejor modelo

In [ ]:
# Validación cruzada del mejor modelo (excluyendo el baseline)
mejores = metrics[metrics["model"] != "Baseline (media)"].sort_values("MAE")
best_name = mejores.iloc[0]["model"]
best_model = modelos_entrenados[best_name]
print("Mejor modelo:", best_name)

cv = cross_val_score(best_model, X, y, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1)
print(f"MAE CV: {(-cv).mean():.4f} ± {cv.std():.4f}")

## 7.4 Predicción vs Real

In [ ]:
# Predicción vs Real (en escala log1p)
y_pred = best_model.predict(X_test)
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.3, edgecolors="none")
lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lim, lim, "r--", lw=2)
plt.xlabel("Real (log1p views)")
plt.ylabel("Predicción (log1p views)")
plt.title(f"{best_name}: Predicción vs Real")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/model_predictions.png", dpi=300, bbox_inches="tight")
plt.show()